In [10]:
# experiment with GSM-8k dataset
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
expset = dataset["train"][:10]
print(expset)
print()
print()
print()
for i in range(10):
    print(expset["question"][i])
    print(expset["answer"][i])
    print()


{'question': ['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?', 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?', 'Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?', 'James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?', 'Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and ther

In [11]:
# train qwen 2.5 0.5 B on GSM 8K dataset
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    tokenizer="Qwen/Qwen2.5-0.5B",
    device=0
)

prompt = "Question: A train travels 60 km in 2 hours. What is its speed?\nAnswer:"

output = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(output[0]["generated_text"])

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3249.90it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: A train travels 60 km in 2 hours. What is its speed?
Answer: 60 km / 2 hours = 30 km / hour
The answer is: 30.

Question: A man bought 80 books at the market price of 120. He sold 40 of them at the market price of 150 and 20 at the market price of a certain amount. What was the selling price of the books he sold at the market price of 150?
Answer: 80 - 4


In [12]:
# From chatgpt

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"  # auto uses your GPU
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2060.30it/s]


In [13]:
# cntnd

from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")

train_data = dataset["train"]
test_data = dataset["test"]

def build_few_shot_prompt(k=5):
    prompt = ""

    for i in range(k):
        q = train_data[i]["question"]
        a = train_data[i]["answer"]

        prompt += f"Question: {q}\n"
        prompt += "Let's think step by step.\n"
        prompt += f"{a}\n\n"

    return prompt

def solve_question(question, few_shot_prompt):
    prompt = (
        few_shot_prompt +
        f"Question: {question}\n"
        "Let's think step by step.\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response[len(prompt):]

few_shot_prompt = build_few_shot_prompt(k=5)

question = test_data[0]["question"]

answer = solve_question(question, few_shot_prompt)

print("QUESTION:\n", question)
print("\nMODEL OUTPUT:\n", answer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


QUESTION:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL OUTPUT:
 Janet’s daily earnings from selling eggs are 16 - 3 - 4 = 7 eggs.
She makes 7 * $2 = $<<7*2=14>>14 every day.
#### 14


In [21]:
# model chat template
import torch, re, random
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"


dataset = load_dataset("gsm8k", "main")

train_data = dataset["train"]
test_data = dataset["test"]

def build_messages(question, k=3):
    messages = [
        {
            "role": "system",
            "content": (
    "You are a precise math problem solver.\n"
    "Solve step by step carefully.\n"
    "Always end your answer with: #### <number>\n"
    "Do not output anything after that."
)
        }
    ]
    indices = random.sample(range(len(train_data)), k)

    # few-shot examples
    for i in indices:
        q = train_data[i]["question"]
        a = train_data[i]["answer"]

        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": a})

    # actual question
    messages.append({"role": "user", "content": question})

    return messages

def chat_llm(question):
    messages = build_messages(question, k=0)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1, # ensure consistent outputs
        do_sample=True
    )

    # ONLY decode new tokens
    generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response

def extract_answer(text):
    match = re.search(r"####\s*(-?\d+\.?\d*)", text)
    if match:
        return match.group(1)

    # fallback
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None


correct = 0
total = 10  # start small


for i in range(total):
    q = test_data[i]["question"]
    gt = extract_answer(test_data[i]["answer"])

    pred_text = chat_llm(q)
    pred = extract_answer(pred_text)

    print(f"\nQ{i}: {q}")
    print("Model:", pred_text)
    print("GT:", gt, "| Pred:", pred)

    if pred == gt:
        correct += 1

print(f"\nAccuracy: {correct}/{total} = {correct/total:.2f}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4608.60it/s]



Q0: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Model: To determine how much Janet makes every day from selling the leftover eggs, we can follow these steps:

1. **Calculate total eggs laid per day:**  
   - Janet's ducks lay 16 eggs per day.

2. **Subtract eggs eaten for breakfast:**  
   - Janet eats 3 eggs for breakfast each morning, so subtract this amount from the total eggs laid:
     \[
     16 \text{ eggs} - 3 \text{ eggs} = 13 \text{ eggs}
     \]

3. **Subtract eggs baked for friends:**  
   - Janet bakes muffins for her friends every day, which means she has 4 muffins left over (since there are no muffins baked). Subtract this amount from the remaining eggs:
     \[
     13 \text{ eggs} - 4 \text{ eggs} = 9 \text{ eggs}
     \]

4. **Calcula